# 03 - Neighborhood "feel" features (quiet, green, access, form, centrality)

Replaces the shop-mix similarity from notebook 02 with features that match what I actually care about:
quiet streets, green space, access to a library/park/groceries at bike and drive scale, moderate density,
and moderate distance to downtown. **No income and no boutique/shop-mix features.** Price will be a budget filter later, not a similarity feature.

Inputs: the parquet cache from notebook 02 (POIs, walk-network nodes/edges, boundaries) plus two new extracts from the same Geofabrik PBF (road classes, park polygons). `data/labels.csv` holds the labeled places.

Test before trusting anything (write the answer down first): does the home-area anchor's nearest-Houston list include West U and Museum District? Do Midtown and Richmond-like places look different?

Similarity here is **weighted Euclidean distance on z-scored features**, not cosine: absolute level matters (a quiet street is not the same as "a bit less busy than average").

In [ ]:
# Kaggle/Colab: uncomment. Locally: pip install in your venv instead.
# %pip install -q geopandas pyarrow pyrosm "shapely>=2" scipy

In [ ]:
import os, warnings
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd, shapely
from shapely import STRtree
from scipy.spatial import cKDTree
from pyproj import Transformer
from pyrosm import OSM
warnings.filterwarnings("ignore")

CACHE = Path("/kaggle/working/cache") if os.path.exists("/kaggle") else Path("../data/interim")
PBF_DIR = CACHE / "pbf"
STATE = {"houston": "texas", "austin": "texas", "seattle": "washington", "atlanta": "georgia"}
CITIES = ["houston"]   # add "austin", "seattle", "atlanta" once Houston looks right
# Approximate downtown points (lat, lon) - VERIFY ON A MAP
DOWNTOWN = {"houston": (29.7604, -95.3698), "austin": (30.2672, -97.7431),
            "seattle": (47.6062, -122.3321), "atlanta": (33.7490, -84.3880)}
SPACING_M, RADIUS_M, NMS_M = 400, 800, 1600
BIG_PARK_HA = 10          # "large park" threshold (Hermann Park is far above this)
MIN_POI_800 = 10          # drop empty land: cell needs this many POIs of any kind within 800 m

def city_crs(bbox):
    lon = (bbox[0] + bbox[2]) / 2
    return f"EPSG:{32600 + int((lon + 180) // 6) + 1}"   # must match notebook 02

def latest(pattern):
    fs = sorted(CACHE.glob(pattern), key=lambda p: p.stat().st_mtime)
    if not fs: raise FileNotFoundError(f"{pattern} not in {CACHE}; run notebook 02 first")
    return fs[-1]

def boundary_file(city):
    # notebook 02 writes a hash-named file for multi-query cities (Houston + enclaves); prefer the newest of those
    fs = sorted(CACHE.glob(f"{city}_boundary_*.geojson"), key=lambda p: p.stat().st_mtime) or [CACHE / f"{city}_boundary.geojson"]
    return fs[-1]

def pbf_path(city):
    name = f"{STATE[city]}-latest.osm.pbf"
    local = PBF_DIR / name
    if local.exists(): return local
    found = next(Path("/kaggle/input").glob(f"**/{name}"), None)   # a PBF saved in an attached dataset
    return found or local

# labels: ../data/labels.csv locally; on Kaggle upload it to /kaggle/working or attach it as a dataset
cands = [Path("../data/labels.csv"), Path("/kaggle/working/labels.csv")] + list(Path("/kaggle/input").glob("**/labels.csv"))
LABELS = next((p for p in cands if p.exists()), None)
assert LABELS is not None, "labels.csv not found"
labels = pd.read_csv(LABELS).dropna(subset=["lat", "lon"]).reset_index(drop=True)
print(labels[["name", "city", "label"]].to_string(index=False))

## 0b. Setup on Kaggle: copy the saved cache out of the read-only dataset, download the PBF only if needed

In [ ]:
import shutil, subprocess, urllib.request
CACHE.mkdir(parents=True, exist_ok=True)

# 1) Attached datasets are read-only under /kaggle/input. Copy the cached parquet/geojson into the working cache.
for src in Path("/kaggle/input").glob("**/*"):
    if src.suffix in (".parquet", ".geojson") and not (CACHE / src.name).exists():
        shutil.copy2(src, CACHE / src.name); print("copied", src.name)

# 2) Road classes and park polygons need the PBF. Skip the big download if those extracts are already cached.
need_pbf = any(not (CACHE / f"{c}_roads_class.parquet").exists() or not (CACHE / f"{c}_major_pts.parquet").exists()
               or not (CACHE / f"{c}_parks.parquet").exists() or not (CACHE / f"{c}_trail_ways.parquet").exists()
               or not (CACHE / f"{c}_libraries.parquet").exists()
               or not (CACHE / f"{c}_buildings.parquet").exists() for c in CITIES)
if need_pbf:
    PBF_DIR.mkdir(exist_ok=True)
    for s in sorted({STATE[c] for c in CITIES}):
        f = PBF_DIR / f"{s}-latest.osm.pbf"
        if f.exists() or next(Path("/kaggle/input").glob(f"**/{s}-latest.osm.pbf"), None): continue
        part = f.with_name(f.name + ".part"); url = f"https://download.geofabrik.de/north-america/us/{s}-latest.osm.pbf"
        print("downloading", url)
        if shutil.which("wget"): subprocess.run(["wget", "-q", "-c", "-O", str(part), url], check=True)
        else: urllib.request.urlretrieve(url, part)
        part.rename(f)
print("cache files:", len(list(CACHE.glob("*.parquet"))), "parquet,", len(list(CACHE.glob("*.geojson"))), "geojson")

## 1. Extra extracts from the PBF (cached): road classes and park polygons

In [ ]:
MAJOR = {"motorway", "motorway_link", "trunk", "trunk_link", "primary", "primary_link", "secondary", "secondary_link"}
FREEWAY = {"motorway", "motorway_link", "trunk", "trunk_link"}

def load_roads(city, bbox, crs):
    fr, fm = CACHE / f"{city}_roads_class.parquet", CACHE / f"{city}_major_pts.parquet"
    if fr.exists() and fm.exists():
        return pd.read_parquet(fr), pd.read_parquet(fm)
    ed = OSM(str(pbf_path(city)), bounding_box=list(bbox)).get_network(network_type="driving").to_crs(crs)
    ed = ed[ed.geometry.notna()].reset_index(drop=True)
    hw = ed["highway"].astype(str)
    mid = ed.geometry.interpolate(0.5, normalized=True)
    roads = pd.DataFrame({"x": mid.x.values, "y": mid.y.values, "len": ed.geometry.length.values,
                          "major": hw.isin(MAJOR).values, "freeway": hw.isin(FREEWAY).values})
    rows = []   # points every ~50 m along major roads, for nearest-distance queries
    for geom, fw in zip(ed.geometry[roads["major"]], roads["freeway"][roads["major"]]):
        for t in np.linspace(0, 1, max(int(geom.length // 50), 1) + 1):
            p = geom.interpolate(t, normalized=True); rows.append((p.x, p.y, bool(fw)))
    pts = pd.DataFrame(rows, columns=["x", "y", "freeway"])
    roads.to_parquet(fr); pts.to_parquet(fm)
    return roads, pts

def load_parks(city, bbox, crs):
    f = CACHE / f"{city}_parks.parquet"
    if f.exists():
        return gpd.read_parquet(f)
    g = OSM(str(pbf_path(city)), bounding_box=list(bbox)).get_data_by_custom_criteria(
        custom_filter={"leisure": ["park", "garden", "nature_reserve"]}, filter_type="keep",
        keep_nodes=False, keep_ways=True, keep_relations=True)
    g = g[g.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].to_crs(crs)
    g = g[["geometry"]].reset_index(drop=True)
    g["area_ha"] = g.geometry.area / 1e4
    g.to_parquet(f)
    return g

import re
# Greenway / exercise-path filter. A path (cycleway, path or footway, minus sidewalks/crossings) counts if ANY of:
#  (a) its name looks like a greenway or trail, (b) it is a standalone cycleway of at least MIN_CYCLEWAY_M,
#  (c) it lies inside a big park (>= BIG_PARK_HA). Everything is applied on each run, so tune and rerun without re-extracting.
GREENWAY_RE = re.compile(r"greenway|trail|bayou|creek|hike|bike|loop|river|lake|rail", re.I)
MIN_CYCLEWAY_M = 300

def load_trails(city, bbox, crs, big_parks):
    f = CACHE / f"{city}_trail_ways.parquet"
    if f.exists():
        ways = gpd.read_parquet(f)
    else:
        g = OSM(str(pbf_path(city)), bounding_box=list(bbox)).get_data_by_custom_criteria(
            custom_filter={"highway": ["cycleway", "path", "footway"]}, filter_type="keep",
            keep_nodes=False, keep_ways=True, keep_relations=False)
        g = g[g.geometry.geom_type.isin(["LineString", "MultiLineString"])].to_crs(crs)
        if "footway" in g:
            g = g[~g["footway"].isin(["sidewalk", "crossing", "access_aisle", "traffic_island", "link"])]
        if "access" in g:
            g = g[~g["access"].isin(["private", "no"])]
        col = lambda k: g[k].fillna("").astype(str).values if k in g else np.full(len(g), "", dtype=object)
        ways = gpd.GeoDataFrame({"name": col("name"), "highway": col("highway"), "length_m": g.geometry.length.values},
                                geometry=g.geometry.values, crs=crs)
        ways.to_parquet(f)
    rp = ways.geometry.representative_point()
    in_park = (shapely.contains_xy(shapely.union_all(big_parks.geometry.values), rp.x.values, rp.y.values)
               if len(big_parks) else np.zeros(len(ways), dtype=bool))
    by_name = ways["name"].str.contains(GREENWAY_RE)
    by_cycleway = (ways["highway"] == "cycleway") & (ways["length_m"] >= MIN_CYCLEWAY_M)
    keep = (by_name | by_cycleway | in_park)
    print(f"{city}: trail ways kept {int(keep.sum())} of {len(ways)} (name {int(by_name.sum())}, long cycleway {int(by_cycleway.sum())}, in big park {int(in_park.sum())})")
    print("  longest kept named trails (m):", ways[keep & (ways["name"] != "")].groupby("name")["length_m"].sum().nlargest(12).round().to_dict())
    rows = []   # points every ~50 m along each kept line
    for geom in ways.geometry[keep]:
        for ln in (geom.geoms if geom.geom_type == "MultiLineString" else [geom]):
            for s in np.arange(0, max(ln.length, 1), 50):
                p = ln.interpolate(s); rows.append((p.x, p.y))
    return pd.DataFrame(rows, columns=["x", "y"])

import re
# OSM has no consistent "public library" tag, so drop libraries that look like school/university/private ones.
NONPUBLIC_RE = re.compile(r"\b(?:school|elementary|middle|high school|academy|college|university|campus|isd|seminary|montessori|church|prep)\b", re.I)
# A name/operator that clearly says "public library / branch / neighborhood library" overrides the school/university words
# (e.g. "Fort Bend County Libraries - University Branch").
PUBLIC_RE = re.compile(r"\b(?:public librar|neighborhood librar|branch|regional librar|county librar|express\b)", re.I)
# Manual overrides by exact name (applied on every run). Edit after reading the kept/dropped lists below.
LIB_KEEP = {"West University Library"}
LIB_DROP = {"M.D. Anderson Library", "HCC WestLoop Library", "HAM-TMC Library", "Woodson Research Center",
            "Lanier Theological Library", "Library Service Center"}

def load_libraries(city, bbox, crs):
    f = CACHE / f"{city}_libraries.parquet"
    if f.exists():
        df = pd.read_parquet(f)
    else:
        g = OSM(str(pbf_path(city)), bounding_box=list(bbox)).get_pois(custom_filter={"amenity": ["library"]}).to_crs(crs)
        cen = g.geometry.centroid
        col = lambda k: g[k].fillna("").astype(str).values if k in g else np.full(len(g), "", dtype=object)
        df = pd.DataFrame({"x": cen.x.values, "y": cen.y.values, "name": col("name"), "operator": col("operator"),
                           "access": col("access"), "tags": col("tags")})
        df.to_parquet(f)
    text = df["name"] + " " + df["operator"] + " " + df["tags"]
    looks_public = (df["name"] + " " + df["operator"]).str.contains(PUBLIC_RE)
    bad_access = df["access"].isin(["private", "no", "customers", "permit", "students"])
    pub = (~text.str.contains(NONPUBLIC_RE) | looks_public) & ~bad_access
    pub = (pub & ~df["name"].isin(LIB_DROP)) | df["name"].isin(LIB_KEEP)
    return df.assign(public=pub.values)

# Housing-type features from OSM building tags. Coverage varies a lot (many buildings are just building=yes),
# so shares are shrunk toward the city average when few residential buildings are tagged nearby.
SMALL_HOME = {"house", "detached", "semidetached_house", "terrace", "townhouse", "bungalow", "duplex"}
LARGE_RES_M2, LARGE_LEVELS = 800, 3   # building=residential counts as multifamily if footprint or height is this big
SHRINK_M2 = 20000.0                   # footprint of "pseudo-evidence" at the city average

def load_buildings(city, bbox, crs):
    f = CACHE / f"{city}_buildings.parquet"
    if f.exists():
        return pd.read_parquet(f)
    g = OSM(str(pbf_path(city)), bounding_box=list(bbox)).get_buildings().to_crs(crs)
    g = g[g.geometry.geom_type.isin(["Polygon", "MultiPolygon"])]
    cen = g.geometry.centroid
    lv = pd.to_numeric(g["building:levels"], errors="coerce").values if "building:levels" in g else np.full(len(g), np.nan)
    df = pd.DataFrame({"x": cen.x.values.astype("float32"), "y": cen.y.values.astype("float32"),
                       "area": g.geometry.area.values.astype("float32"), "levels": lv.astype("float32"),
                       "building": g["building"].astype(str).astype("category").values})
    df.to_parquet(f)
    return df

def classify(b):
    t = b["building"].astype(str)
    mf = (t == "apartments") | ((t == "residential") & ((b["area"] >= LARGE_RES_M2) | (b["levels"] >= LARGE_LEVELS)))
    small = t.isin(SMALL_HOME)
    oth = (t == "residential") & ~mf
    return mf.values, small.values, oth.values, (t == "yes").values

def tree(xy): return cKDTree(xy) if len(xy) else None

D = {}
for c in CITIES:
    poly = gpd.read_file(boundary_file(c)).geometry.iloc[0]
    crs = city_crs(poly.bounds)
    poi = pd.read_parquet(latest(f"{c}_pois_pbf_*.parquet"))
    nodes = pd.read_parquet(latest(f"{c}_nodes_pbf_*.parquet"))
    edges = pd.read_parquet(latest(f"{c}_edges_pbf_*.parquet"))
    roads, mpts = load_roads(c, poly.bounds, crs)
    parks = load_parks(c, poly.bounds, crs)
    trails = load_trails(c, poly.bounds, crs, parks[parks["area_ha"] >= BIG_PARK_HA])
    libs = load_libraries(c, poly.bounds, crs)
    bld = load_buildings(c, poly.bounds, crs)
    mf, small, oth, yes = classify(bld)
    ar = bld["area"].values.astype("float64")
    res_tot = (ar * (mf | small | oth)).sum()
    bld_pack = dict(bldtree=tree(bld[["x", "y"]].values), a_mf=ar * mf, a_small=ar * small, a_oth=ar * oth,
                    a_yes=ar * yes, a_all=ar, prior_mf=(ar * mf).sum() / max(res_tot, 1),
                    prior_small=(ar * small).sum() / max(res_tot, 1))
    big = parks[parks["area_ha"] >= BIG_PARK_HA]
    def pts(tags, p=poi):
        q = p[p["tag"].isin(tags)]; return tree(q[["x", "y"]].values)
    D[c] = dict(
        poly=poly, crs=crs,
        poi_all=tree(poi[["x", "y"]].values),
        library=tree(libs.loc[libs["public"], ["x", "y"]].values), market=pts(["shop=supermarket"]), hospital=pts(["amenity=hospital"]),
        school=pts(["amenity=school"]), eat=pts(["amenity=restaurant", "amenity=cafe", "amenity=fast_food"]),
        ntree=cKDTree(nodes[["x", "y"]].values), ndeg=nodes["deg"].values,
        etree=cKDTree(edges[["x", "y"]].values), elen=edges["len"].values,
        rtree=cKDTree(roads[["x", "y"]].values), rlen=roads["len"].values, rmaj=roads["major"].values,
        majtree=tree(mpts[["x", "y"]].values), fwtree=tree(mpts.loc[mpts["freeway"], ["x", "y"]].values),
        pgeom=parks.geometry.values, ptree=STRtree(parks.geometry.values),
        btree=STRtree(big.geometry.values) if len(big) else None,
        trtree=tree(trails[["x", "y"]].values), libs_df=libs, **bld_pack)
    print(c, "| POIs", len(poi), "| roads", len(roads), "| parks", len(parks), f"({len(big)} >= {BIG_PARK_HA} ha)", "| trail pts", len(trails),
          "| public libraries", int(libs["public"].sum()), "of", len(libs),
          "| buildings", len(bld), f"({yes.mean():.0%} tagged only 'yes', {(mf | small | oth).mean():.0%} residential-tagged)")

### Check the library filter. Fix mistakes by adding exact names to `LIB_DROP` / `LIB_KEEP` (cell above), then rerun from that cell.

In [ ]:
for c in CITIES:
    l = D[c]["libs_df"]
    nm = lambda m: sorted(l.loc[m, "name"].replace("", "(unnamed)"))
    print(f"== {c}: kept {int(l['public'].sum())}, dropped {int((~l['public']).sum())}")
    print("DROPPED:", nm(~l["public"]))
    print("KEPT:   ", nm(l["public"]))

## 2. Features per point (works for labeled places and for grid cells)

In [ ]:
GROUPS = {
    "quiet":    ["major_road_share", "dist_major", "dist_freeway"],
    "green":    ["park_frac", "dist_park", "dist_bigpark", "dist_trail", "trail_km_1500"],
    "vicinity": ["dist_library", "n_library_3km", "n_market_3km", "n_hospital_5km", "n_school_1500", "n_eat_1500"],
    "form":     ["intersection_density", "mean_edge_len", "log_poi_800"],
    "center":   ["dist_downtown_km"],
    "housing":  ["mf_share", "small_home_share"],
}
FEATS = [f for cols in GROUPS.values() for f in cols]
# Tune these per anchor. Walkability-type "form" features get low weight on purpose.
WEIGHTS = {"quiet": 1.0, "green": 1.0, "vicinity": 1.0, "form": 0.5, "center": 0.5, "housing": 0.5}
# Hard filter on search results: exclude cells where large apartment complexes make up more than this share of the
# residential footprint nearby (None = off). Try 0.5-0.6 once the coverage check below looks trustworthy.
MAX_MF_SHARE = None
W = np.array([WEIGHTS[g] / np.sqrt(len(cols)) for g, cols in GROUPS.items() for _ in cols])
LOGCOLS = [f for f in FEATS if f.startswith(("dist_", "n_")) or f in ("intersection_density", "mean_edge_len", "trail_km_1500")]

def to_xy(city, lat, lon):
    return Transformer.from_crs("EPSG:4326", D[city]["crs"], always_xy=True).transform(lon, lat)

def to_latlon(city, x, y):
    lon, lat = Transformer.from_crs(D[city]["crs"], "EPSG:4326", always_xy=True).transform(x, y)
    return lat, lon

def near(t, pts, cap):
    if t is None: return np.full(len(pts), float(cap))
    return np.minimum(t.query(np.c_[shapely.get_x(pts), shapely.get_y(pts)])[0], cap)

def pdist(t, pts, cap):
    if t is None: return np.full(len(pts), float(cap))
    _, dist = t.query_nearest(pts, return_distance=True, all_matches=False)
    return np.minimum(dist, cap)

def count(t, P, r):
    return np.zeros(len(P)) if t is None else t.query_ball_point(P, r, return_length=True).astype(float)

def bldg_stats(c, P, chunk=500):
    # footprint area (m²) of each building class within RADIUS_M; chunked because there are ~1M buildings
    d = D[c]; out = np.zeros((len(P), 5))
    for s in range(0, len(P), chunk):
        for k, idx in enumerate(d["bldtree"].query_ball_point(P[s:s + chunk], RADIUS_M)):
            if idx:
                out[s + k] = [d["a_mf"][idx].sum(), d["a_small"][idx].sum(), d["a_oth"][idx].sum(),
                              d["a_yes"][idx].sum(), d["a_all"][idx].sum()]
    return pd.DataFrame(out, columns=["mf", "small", "other_res", "untagged", "all"])

def housing_feats(c, S):
    d = D[c]; res = S["mf"] + S["small"] + S["other_res"]
    return pd.DataFrame({"mf_share": (S["mf"] + SHRINK_M2 * d["prior_mf"]) / (res + SHRINK_M2),
                         "small_home_share": (S["small"] + SHRINK_M2 * d["prior_small"]) / (res + SHRINK_M2)})

def compute(c, P):
    d = D[c]; n = len(P); area_km2 = np.pi * (RADIUS_M / 1000) ** 2
    pts = shapely.points(P[:, 0], P[:, 1])
    # quiet: share of nearby road length that is major, distance to major/freeway
    tot, maj = np.zeros(n), np.zeros(n)
    for k, idx in enumerate(d["rtree"].query_ball_point(P, RADIUS_M)):
        if idx:
            l = d["rlen"][idx]; tot[k] = l.sum(); maj[k] = l[d["rmaj"][idx]].sum()
    # green: park area share within 800 m
    bufs = shapely.buffer(pts, RADIUS_M); pf = np.zeros(n)
    for k, b in enumerate(bufs):
        ids = d["ptree"].query(b)
        if len(ids): pf[k] = shapely.area(shapely.intersection(d["pgeom"][ids], b)).sum() / b.area
    # form: intersection density, mean edge length
    inter, mel = np.zeros(n), np.zeros(n)
    for k, idx in enumerate(d["ntree"].query_ball_point(P, RADIUS_M)):
        if idx: inter[k] = (d["ndeg"][idx] >= 3).sum() / area_km2
    for k, idx in enumerate(d["etree"].query_ball_point(P, RADIUS_M)):
        if idx: mel[k] = d["elen"][idx].mean()
    H = housing_feats(c, bldg_stats(c, P))
    # centrality
    tx = Transformer.from_crs("EPSG:4326", d["crs"], always_xy=True).transform(DOWNTOWN[c][1], DOWNTOWN[c][0])
    out = {
        "major_road_share": np.where(tot > 0, maj / np.maximum(tot, 1), 0.0),
        "dist_major": near(d["majtree"], pts, 5000), "dist_freeway": near(d["fwtree"], pts, 8000),
        "park_frac": pf, "dist_park": pdist(d["ptree"], pts, 3000), "dist_bigpark": pdist(d["btree"], pts, 8000),
        "dist_trail": near(d["trtree"], pts, 5000), "trail_km_1500": count(d["trtree"], P, 1500) * 50 / 1000,
        "mf_share": H["mf_share"].values, "small_home_share": H["small_home_share"].values,
        "dist_library": near(d["library"], pts, 8000), "n_library_3km": count(d["library"], P, 3000),
        "n_market_3km": count(d["market"], P, 3000), "n_hospital_5km": count(d["hospital"], P, 5000),
        "n_school_1500": count(d["school"], P, 1500), "n_eat_1500": count(d["eat"], P, 1500),
        "intersection_density": inter, "mean_edge_len": mel,
        "log_poi_800": np.log1p(count(d["poi_all"], P, RADIUS_M)),
        "dist_downtown_km": np.hypot(P[:, 0] - tx[0], P[:, 1] - tx[1]) / 1000,
    }
    return pd.DataFrame(out)[FEATS]

def make_grid(c):
    d = D[c]; w, s, e, n = d["poly"].bounds
    t = Transformer.from_crs("EPSG:4326", d["crs"], always_xy=True)
    (x0, y0), (x1, y1) = t.transform(w, s), t.transform(e, n)
    X, Y = np.meshgrid(np.arange(x0, x1, SPACING_M), np.arange(y0, y1, SPACING_M))
    poly_xy = gpd.GeoSeries([d["poly"]], crs=4326).to_crs(d["crs"]).iloc[0]
    m = shapely.contains_xy(poly_xy, X.ravel(), Y.ravel())
    return np.c_[X.ravel(), Y.ravel()][m]

GRID = {}
for c in CITIES:
    P = make_grid(c); F = compute(c, P)
    GRID[c] = dict(P=P, F=F, ok=(F["log_poi_800"] >= np.log1p(MIN_POI_800)).values)
    print(c, "cells:", len(P), "usable:", int(GRID[c]["ok"].sum()))

## 3. Labeled places: raw values and percentile within their city (usable cells)

In [ ]:
rows, names, diag = [], [], []
for _, r in labels.iterrows():
    if r["city"] not in D: continue     # e.g. Richmond is outside the Houston boundary
    x, y = to_xy(r["city"], r["lat"], r["lon"])
    rows.append(compute(r["city"], np.array([[x, y]])).iloc[0]); names.append(r["name"])
    diag.append(bldg_stats(r["city"], np.array([[x, y]])).iloc[0])
L = pd.DataFrame(rows, index=names).astype(float)
pd.set_option("display.width", 220, "display.max_columns", 30)
print(L.round(2).T.to_string())

# Coverage check for the housing features: how much of the nearby footprint is tagged residential vs just 'yes'?
Dg = pd.DataFrame(diag, index=names); res_m2 = Dg["mf"] + Dg["small"] + Dg["other_res"]
print("\nBuilding tag coverage within 800 m (low tagged share = housing features are shrunk toward the city average):")
print(pd.DataFrame({"mf_m2": Dg["mf"].round(), "small_home_m2": Dg["small"].round(), "other_res_m2": Dg["other_res"].round(),
                    "untagged_yes_m2": Dg["untagged"].round(),
                    "residential_tagged_share": (res_m2 / Dg["all"].clip(lower=1)).round(2)}).to_string())

print("\nPercentile of each place among usable cells of its city (0 = lowest, 1 = highest):")
pct = {}
for n_, r in labels.set_index("name").loc[L.index].iterrows():
    G = GRID[r["city"]]["F"][GRID[r["city"]]["ok"]]
    pct[n_] = [(G[f] < L.loc[n_, f]).mean() for f in FEATS]
print(pd.DataFrame(pct, index=FEATS).round(2).to_string())

**Look at this table before the search.** Do the liked places (home area, West U, Museum District) share a pattern on quiet / green / vicinity?
Do Summer St and Midtown differ (more major roads, higher intersection density)? If the labeled places do not separate on these features, searching will not help yet.

## 4. Search: nearest cells to a labeled anchor (one query per anchor, NMS, biggest gaps)

In [ ]:
def transform(F):
    T = F.copy()
    for c in LOGCOLS: T[c] = np.log1p(T[c])
    return T

ALL = pd.concat([GRID[c]["F"][GRID[c]["ok"]] for c in CITIES])
TA = transform(ALL); MU, SD = TA.mean(), TA.std() + 1e-9
def Z(F): return ((transform(F) - MU) / SD).clip(-3, 3).values

def search(anchor, target, k=8):
    a = labels[labels["name"] == anchor].iloc[0]
    za = Z(L.loc[[anchor]])[0]
    T = GRID[target]; ZT = Z(T["F"])
    d = np.sqrt((((ZT - za) * W) ** 2).sum(1)); d[~T["ok"]] = 1e9
    if MAX_MF_SHARE is not None: d[T["F"]["mf_share"].values > MAX_MF_SHARE] = 1e9
    if a["city"] == target:   # exclude the anchor's own neighborhood
        ax, ay = to_xy(a["city"], a["lat"], a["lon"])
        d[np.hypot(T["P"][:, 0] - ax, T["P"][:, 1] - ay) < NMS_M] = 1e9
    chosen = []
    for j in np.argsort(d):
        if d[j] >= 1e9: break
        if all(np.hypot(*(T["P"][j] - T["P"][c])) >= NMS_M for c in chosen): chosen.append(j)
        if len(chosen) == k: break
    out = []
    for j in chosen:
        lat, lon = to_latlon(target, *T["P"][j])
        gap = np.argsort(-np.abs((ZT[j] - za) * W))[:3]
        out.append(dict(dist=round(float(d[j]), 2), lat=round(lat, 5), lon=round(lon, 5),
                        map=f"https://maps.google.com/?q={lat:.5f},{lon:.5f}",
                        biggest_gaps=", ".join(np.array(FEATS)[gap])))
    return pd.DataFrame(out)

pd.set_option("display.max_colwidth", 90)
for anchor in L.index:
    if labels.set_index("name").loc[anchor, "label"] not in ("like", "like_conditional"): continue
    for target in CITIES:
        print(f"\n=== {anchor}  ->  {target} (lower dist = closer) ===")
        print(search(anchor, target).to_string(index=False))

## 5. Not built yet (next steps, in order)
1. **ACS census features** (share of households with children, 65+ share, owner share, median rent) by tract, joined to cells. Rent is a *filter*, the family features go in similarity. Needs a Census API call and tract shapes (TIGER/Line).
2. **Tree canopy** (NLCD/USFS raster) for "green" beyond park polygons.
3. **Negative anchors**: use `dislike` rows (needs the previous apartment's location, and a grid that covers Richmond) to push down cells that look more like a disliked place than a liked one.
4. Add Austin / Seattle / Atlanta to `CITIES`. Compare the within-Houston result first; only then trust cross-city.
5. Per-anchor weights (`WEIGHTS`) from the labeled places, as in the CLAUDE.md few-shot plan.

## 6. Export the cache (old + new extracts) as a clean folder for a new dataset version

In [ ]:
EXPORT = Path("/kaggle/working/export") if os.path.exists("/kaggle") else CACHE.parent / "export"
EXPORT.mkdir(parents=True, exist_ok=True)
files = [p for p in CACHE.iterdir() if p.is_file() and p.suffix in (".parquet", ".geojson")]
for p in files: shutil.copy2(p, EXPORT / p.name)
print(f"{len(files)} files, {sum(p.stat().st_size for p in files) / 1e6:.0f} MB in {EXPORT}")
for c in CITIES:
    for n in (f"{c}_roads_class", f"{c}_major_pts", f"{c}_parks", f"{c}_trail_ways", f"{c}_libraries", f"{c}_buildings"):
        print("ok      " if (EXPORT / f"{n}.parquet").exists() else "MISSING ", n)
print("Next: Data -> Output -> export folder -> update the cache dataset (New Version).")